# Idea 6 of the ideas planner: map self-similarity ("twins") as a signal of how ambiguous a place is
Mechanism: cut the satellite tile at a candidate place, correlate it with the 5.5 km disc around it by the pipeline's own coarse search (edge-direction ZNCC, satellite against satellite, no thermal noise, no learning). Places with many equally good twins are ambiguous, so a few inliers there should not be trusted.
Test (fixed before looking): per photo, a RIGHT candidate (best-inlier candidate within 1 sigma of the truth, >= 3 inliers) and a HARD WRONG candidate (best-inlier candidate more than 4 sigma from the truth, >= 3 inliers) from the 27-query dump. Features of each: twin count (coarse peaks away from the place with score >= own score - 0.05 / - 0.02), and the margin own score minus best other peak. Dead if AUC (wrong vs right) is under 0.7 on either domain. Also the seconds of the extra correlation (the cost of using it): one coarse search per answered photo.
Settings by environment: GATE_DATA (official / nudt), DUMPS (comma list of rich_dump folders), POOL_PHOTOS, OUT.


In [1]:
import os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)

DATA = os.environ.get("GATE_DATA", "official")
MAP = "new"
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))   # same pool and setup seed as Test_Iteration_3, so NUDT seed 101 / 202 pick the same photos
WIDE_HEADINGS = [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0]
WIDE_SCALES = [0.95, 1.0, 1.05]
K_PER_QUERY, M_PER_QUERY = 50, 10
MATCHER_CONFIDENCE, LOFTR_COARSE_THRESHOLD = 0.22, 0.21
MIN_INLIERS, GAMMA = 8, 1.0
NUDT_SEED = int(os.environ.get("NUDT_SEED", "101"))
N_NUDT = int(os.environ.get("N_NUDT", "300"))
EXCLUDE_SEEDS = [int(s) for s in os.environ.get("EXCLUDE_SEEDS", "").split(",") if s.strip()]
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))
EXCLUDE_N = int(os.environ.get("EXCLUDE_N", "300"))   # size of the excluded seeds' samples (seed 101 picked 300; this reproduces Test_Iteration_3's seed 202 photos exactly)
MATCHER = os.environ.get("MATCHER", "loftr")          # "loftr" (the pipeline) or "xoftr" (MINIMA XoFTR, a swap-in with the same plumbing)
XOFTR_COARSE, XOFTR_FINE = float(os.environ.get("XOFTR_COARSE", "0.3")), float(os.environ.get("XOFTR_FINE", "0.1"))
FRESH_EXCLUDE_POOL = int(os.environ.get("FRESH_EXCLUDE_POOL", "0"))   # >0 (NUDT, with POOL_PHOTOS >= all frames): skip every photo of the usual
# pool of this size (setup seed 7), so the sample is from photos no earlier test has used
EXCLUDE_IDS_FILE = os.environ.get("EXCLUDE_IDS_FILE")   # optional csv with a photo column: never draw these photos (an earlier fresh sample)
PHOTOS_FILE = os.environ.get("PHOTOS_FILE")           # optional csv with a photo column: only these photos
print(MATCHER, DATA, "seed", NUDT_SEED, "n", N_NUDT, "exclude", EXCLUDE_SEEDS, "limit", N_LIMIT)

loftr official seed 101 n 300 exclude [] limit 0


In [2]:
import torch, gc, math, time
from train.checks import replay
from train.checks.rich_dump import RichDumpPipeline
from image_processing.map_levels import map_piece
from semantic_rerank.dino_rerank import shortlist_peaks
from train.coarse_methods.inputs import make_inputs
pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=GAMMA, k_peaks=K_PER_QUERY, m_fine=M_PER_QUERY,
                            matcher="loftr_minima", min_inliers=MIN_INLIERS, ransac_pixels=None, matcher_confidence=MATCHER_CONFIDENCE, satellite=MAP,
                            map_margin_m=None, loftr_coarse_threshold=LOFTR_COARSE_THRESHOLD, dino_weight=None, headings=[0.0], scale_factors=[1.0])
t0 = time.time(); pipeline.setup(POOL_PHOTOS, 7); print("setup %.0f s" % (time.time() - t0))
T7 = ROOT / "tests/Test_Iteration_7/results"


level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 114 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 130 s


In [3]:
ALL = [(h, s) for s in (0.95, 1.0, 1.05) for h in (-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0)]
def candidates(photo):
    ids = replay.query_ids(photo, ALL)
    c = replay.pooled(photo, ids)
    tx, ty = photo["truth"]; sig = photo["sigma"]
    err = np.hypot(c["x"] - tx, c["y"] - ty) / sig
    ok = c["inliers"] >= 3
    out = {}
    right = np.where(ok & (err <= 1.0))[0]; wrong = np.where(ok & (err > 4.0))[0]
    if len(right): k = right[np.argmax(c["inliers"][right])]; out["right"] = (c["x"][k], c["y"][k], c["inliers"][k], err[k])
    if len(wrong): k = wrong[np.argmax(c["inliers"][wrong])]; out["wrong"] = (c["x"][k], c["y"][k], c["inliers"][k], err[k])
    return out
def twin_features(x, y, level, sigma):
    canvas = map_piece(pipeline.level_images, level, x, y)
    valid = np.ones(canvas.shape, bool)
    clock = time.perf_counter()
    inputs = make_inputs(canvas, valid, level, (x, y), pipeline.families, pipeline.names)
    score_map = pipeline.coarse_method(inputs)
    peaks = shortlist_peaks(score_map, inputs["geometry"], 50)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    seconds = time.perf_counter() - clock
    real = peaks[:, 1] < 1e8
    p = peaks[real]
    dist = np.hypot(p[:, 1] - x, p[:, 2] - y) / sigma
    own = np.where(dist <= 4.0)[0]
    own_score = float(p[own, 0].max()) if len(own) else float(p[:, 0].max())
    far = p[dist > 4.0]
    far_scores = far[:, 0] if len(far) else np.array([-1.0])
    return dict(own_score=own_score, best_other=float(far_scores.max()), margin=own_score - float(far_scores.max()),
                twins05=int((far_scores >= own_score - 0.05).sum()), twins02=int((far_scores >= own_score - 0.02).sum()), seconds=seconds, self_found=int(len(own) > 0))


In [4]:
rows = []
for tag in os.environ["DUMPS"].split(","):
    dump = replay.load_set(T7 / "rich_dump" / tag)
    for pid, photo in dump.items():
        for kind, (x, y, inl, err) in candidates(photo).items():
            f = twin_features(float(x), float(y), photo["level"], photo["sigma"])
            rows.append(dict(set=tag, photo=pid, kind=kind, height=photo["height"], inliers=float(inl), err_sigma=float(err), **f))
        torch.cuda.empty_cache()
    print(tag, "done", time.time() - t0, flush=True)
Z = pd.DataFrame(rows); Z.to_csv(T7 / ("idea6_twins_%s.csv" % os.environ["OUT"]), index=False); print(len(Z), "candidates")


official150 done 184.03116989135742


235 candidates


In [5]:
from sklearn.metrics import roc_auc_score
def auc(g, col, sign):
    y = (g.kind == "wrong").astype(int)
    return roc_auc_score(y, sign * g[col]) if y.nunique() == 2 else float("nan")
res = []
for name, g in [("all " + os.environ["OUT"], Z)] + [("set " + s, Z[Z.set == s]) for s in Z.set.unique()] + [("inliers 3-11", Z[Z.inliers <= 11])]:
    res.append(dict(group=name, n_right=int((g.kind == "right").sum()), n_wrong=int((g.kind == "wrong").sum()),
                    auc_twins05=auc(g, "twins05", 1), auc_twins02=auc(g, "twins02", 1), auc_margin=auc(g, "margin", -1), auc_inliers=auc(g, "inliers", -1)))
S = pd.DataFrame(res).round(3); S.to_csv(T7 / ("idea6_twins_auc_%s.csv" % os.environ["OUT"]), index=False); print(S.to_string())
print("extra correlation seconds: mean %.2f, median %.2f, p90 %.2f (%d runs)" % (Z.seconds.mean(), Z.seconds.median(), Z.seconds.quantile(0.9), len(Z)))
print(Z.groupby("kind")[["twins05", "twins02", "margin", "own_score", "best_other", "self_found"]].mean().round(3))


             group  n_right  n_wrong  auc_twins05  auc_twins02  auc_margin  auc_inliers
0     all official      145       90          0.5          0.5       0.642        0.963
1  set official150      145       90          0.5          0.5       0.642        0.963
2     inliers 3-11       25       89          0.5          0.5       0.552        0.796
extra correlation seconds: mean 0.18, median 0.11, p90 0.47 (235 runs)
       twins05  twins02  margin  own_score  best_other  self_found
kind                                                              
right      0.0      0.0   0.721      0.784       0.064         1.0
wrong      0.0      0.0   0.646      0.715       0.069         1.0
